# 04 · Execution, runtime guardrails and step records

**Role of this layer.** The executor is the only code that calls capability
functions. It re-validates the plan (it does not trust its caller), refuses to
start while any BLOCK exists or any APPROVAL issue is unapproved, then runs the
steps in order with the runtime guardrails that need data:

| code | severity | checks |
|---|---|---|
| R01 | BLOCK | snapshot file hash still matches the catalog |
| R02 | BLOCK | enough observations for the capability (e.g. 3 x window) |
| R03 | WARN | more than 5% of a raw series missing |
| R04 | WARN | an inner join drops more than 10% of an input |
| R05 | WARN | the flagged group of a comparison or test has fewer than 5 distinct episodes |

Every step produces a **step record**: parameters used (defaults filled), input
hashes, output hash, sample size and range, duration, issues.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import load_catalog
from quantlab.executor import execute_plan
from quantlab.guardrails import fill_defaults
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.plan import Plan

catalog = load_catalog(settings.catalog_path)


def scenario(scenario_id):
    s = SCENARIOS_BY_ID[scenario_id]
    return fill_defaults(Plan(plan_id="demo", request=s.request, **s.plan))


plan_1 = scenario("volatility_flag_success")
refused = execute_plan(plan_1, catalog, approved_issue_ids=set())
print(refused.status, "-", refused.error)

blocked - Approval required for: G10:s3:return_method, G10:s6:return_method


With the approval granted, scenario 1 runs on the real NASDAQ snapshot:

In [3]:
approvals = {"G10:s3:return_method", "G10:s6:return_method"}  # returns and forward returns
result = execute_plan(plan_1, catalog, approved_issue_ids=approvals)
print("status:", result.status, f"({result.duration_ms:.0f} ms)")
pd.DataFrame(
    [
        {
            "step": r.step_id,
            "code": r.capability_code,
            "capability": r.capability,
            "params": r.params,
            "n_obs": r.n_obs,
            "range": f"{r.start}..{r.end}",
            "ms": r.duration_ms,
            "output sha256": r.output_hash[:12],
        }
        for r in result.step_records
    ]
)

status: completed (118 ms)


,step,code,capability,params,n_obs,range,ms,output sha256
0,s1,C01,load_series,{'dataset_id': 'NASDAQCOM'},14042,1971-02-05..2026-10-15,52.69,19533e0349f1
1,s2,C02,resample,"{'freq': 'W', 'how': 'last'}",2906,1971-02-05..2026-10-09,23.03,0c44f38c8ce1
2,s3,C03,compute_returns,"{'method': 'log', 'periods': 1}",2905,1971-02-12..2026-10-09,9.45,01837c38b955
3,s4,C06,rolling_volatility,"{'window': 12, 'annualize': True}",2894,1971-04-30..2026-10-09,9.19,2bf94759d74c
4,s5,C07,flag_regime,"{'quantile': 0.9, 'method': 'expanding', 'min_periods': 52}",2842,1972-04-28..2026-10-09,7.78,d9d8384c0602
5,s6,C08,forward_return,{'method': 'simple'},2905,1971-02-05..2026-10-02,9.29,b4b3370e2ab0
6,s7,C09,compare_groups,{},2841,1972-04-28..2026-10-02,2.83,60848b819f14
7,s8,C10,hac_mean_difference,{},2841,1972-04-28..2026-10-02,3.94,3efe82a2d657


The last step holds the flags: True for weeks of unusually high volatility.

In [4]:
flags = result.artifacts["s5"].value
print(f"{int(flags.sum())} of {len(flags)} weeks flagged")
flags[flags].tail()

446 of 2842 weeks flagged


date
2025-05-23    True
2025-05-30    True
2025-06-06    True
2025-06-13    True
2025-06-20    True
Name: NASDAQCOM, dtype: bool

The last step is the significance test: the next week's simple return after flagged
weeks versus after normal weeks, with Newey-West standard errors (all its numbers are in one table):

In [5]:
pd.Series(result.artifacts["s8"].value, name="s8 hac_mean_difference")

diff_means             -0.000209
mean_normal             0.002302
se_hac                  0.002229
t_stat                 -0.093543
p_value                 0.925472
ci_low                 -0.004578
ci_high                 0.004161
n_obs                       2841
n_flagged                    446
n_normal                    2395
n_episodes_flagged            40
maxlags_used                   8
horizon                        1
start                 1972-04-28
end                   2026-10-02
Name: s8 hac_mean_difference, dtype: object

**R02**: a 250-month window on monthly data does not have 3 x 250 observations.

In [6]:
from quantlab.plan import Step

plan = fill_defaults(
    Plan(
        plan_id="demo",
        request="monthly volatility",
        dataset_ids=["NASDAQCOM"],
        steps=[
            Step(id="s1", capability="load_series", params={"dataset_id": "NASDAQCOM"}),
            Step(id="s2", capability="resample", inputs={"series": "s1"}, params={"freq": "M"}),
            Step(
                id="s3",
                capability="compute_returns",
                inputs={"series": "s2"},
                params={"method": "simple"},
            ),
            Step(
                id="s4",
                capability="rolling_volatility",
                inputs={"series": "s3"},
                params={"window": 250},
            ),
        ],
    )
)
blocked = execute_plan(plan, catalog, {"G10:s3:return_method"})
print(blocked.status, "-", blocked.error)

blocked - Step s4 (rolling_volatility): rolling_volatility has 667 usable observations; it needs 750 (at least 3 x window observations).


**R04** fires on the real data for the 10y-2y spread: the 2-year yield starts in
1976 and the 10-year in 1962, so the spread silently loses 22% of the 10-year
history. The run completes, with the warning recorded.

In [7]:
spread = fill_defaults(
    Plan(
        plan_id="demo",
        request="10y-2y spread",
        steps=[
            Step(id="s1", capability="load_series", params={"dataset_id": "DGS10"}),
            Step(id="s2", capability="load_series", params={"dataset_id": "DGS2"}),
            Step(id="s3", capability="spread", inputs={"a": "s1", "b": "s2"}),
        ],
    )
)
result_e = execute_plan(spread, catalog)
print(result_e.status)
for issue in result_e.issues:
    print(f"[{issue.severity}] {issue.code}: {issue.message}")

completed
[WARN] R04: Aligning the inputs kept 12590 of 16182 rows of 'a' (22% dropped).


Determinism: running the same plan twice gives identical output hashes.

In [8]:
again = execute_plan(spread, catalog)
[r.output_hash for r in again.step_records] == [r.output_hash for r in result_e.step_records]

True